# ClauseCheck
### Agentic RAG for Health Insurance Claim Adjudication

**Generative AI — Assignment 3, Group Mini-Project**

Gundeep Singh · Aditya Bakhale · Raghavendra Prasad Shetty
AI & Data Science, Jio Institute

---

Whether a hospitalisation claim is payable, and for how much, depends on a handful of clauses
scattered across a 50-80 page policy wording: waiting periods, permanent exclusions, room rent
capping, procedure sub-limits and co-payment. The answer is not in any model's parametric
knowledge, because it is specific to one insurer's wording and one claim. That makes it a
retrieval problem. The fact that a single claim needs several separate clause lookups plus a
calculation is what makes it an agent rather than a chain.

```
                    policy wording PDFs
                            |
                    clause-aware chunking
                  +---------+---------+
                  |                   |
            Chroma (dense)      BM25 (keyword)
                  |                   |
                  +---------+---------+
                            | reciprocal rank fusion
                     cross-encoder rerank
                            |
  parse_and_plan -> retrieve -> grade -+-> calculate -> adjudicate
                        ^              |
                        +--- retry ----+
```

**The design decision that matters.** The verdict and the rupee figure come from a deterministic
Python rules engine, not from the model. The model finds the clauses and reads the numbers out
of them; it then explains a decision it did not make. The same claim and the same clauses always
give the same answer, and every figure can be traced to a clause on a page.

## 1. Initial Setup

### Environment (VSCode)

Packages are installed **once, from a terminal**, into the virtual environment that this notebook's kernel uses (see `requirements.txt` and the run steps). Do not install from inside the notebook: `%pip` fails in a `uv` venv, and it can install into a different Python than the kernel.

The cell below only *checks* that the kernel, packages, data folder, API key and Ollama are ready.

In [ ]:
import sys, shutil, subprocess, importlib.util
from pathlib import Path

print("kernel python :", sys.executable)
print("working dir   :", Path.cwd())

problems = []

# 1. packages
required = ["langchain", "langchain_classic", "langchain_community", "langchain_core", "langgraph",
            "langchain_chroma", "langchain_ollama", "langchain_groq", "rank_bm25",
            "sentence_transformers", "pypdf", "gradio", "dotenv", "pandas"]
missing = [m for m in required if importlib.util.find_spec(m) is None]
if missing:
    problems.append(f"missing packages {missing}: run  pip install -r requirements.txt  in the "
                    "terminal of the venv this kernel uses, then restart the kernel")

# 2. data folder
for p in ["data/policies/manifest.json", "data/gold/claims.json"]:
    if not Path(p).exists():
        problems.append(f"missing {p} (keep the data/ folder next to this notebook)")

# 3. Groq key (from .env or the environment; the notebook prompts if neither has it)
if not (Path(".env").exists() or __import__("os").environ.get("GROQ_API_KEY")):
    print("note: no .env file and no GROQ_API_KEY set; you will be asked for the key below")

# 4. Ollama + embedding model
if shutil.which("ollama") is None:
    problems.append("Ollama is not installed or not on PATH (https://ollama.com/download)")
else:
    listing = subprocess.run(["ollama", "list"], capture_output=True, text=True)
    if listing.returncode != 0:
        problems.append("Ollama is installed but not running: start the Ollama app or run  ollama serve")
    elif "nomic-embed-text" not in listing.stdout:
        problems.append("embedding model missing: run  ollama pull nomic-embed-text")

print("\nREADY" if not problems else "\nFIX THESE FIRST:\n - " + "\n - ".join(problems))

In [ ]:
import os, re, json, time, getpass, warnings
from pathlib import Path
from uuid import uuid4
from dataclasses import dataclass, field
from datetime import datetime, date
from typing import Optional, Literal, Any

import pandas as pd
import gradio as gr
from dotenv import load_dotenv
from IPython.display import display, Markdown, Image

from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.retrievers import BM25Retriever

# LangChain 1.x moved the classic retrievers into the langchain-classic package.
# The fallback keeps the notebook working on an older 0.3.x install too.
try:
    from langchain_classic.retrievers import EnsembleRetriever, ContextualCompressionRetriever
    from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
except ImportError:
    from langchain.retrievers import EnsembleRetriever, ContextualCompressionRetriever
    from langchain.retrievers.document_compressors import CrossEncoderReranker

from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import START, StateGraph, END
from typing_extensions import List, TypedDict
from pydantic import BaseModel, Field

In [ ]:
warnings.filterwarnings('ignore')
load_dotenv(override=True)

In [ ]:
#Check for Groq API Key
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ API Key: ")

print("Groq key loaded:", bool(os.environ.get("GROQ_API_KEY")))

In [ ]:
# Paths and every hyperparameter we tune, in one place.
DATA_DIR      = Path("data")
POLICY_DIR    = DATA_DIR / "policies"
GOLD_PATH     = DATA_DIR / "gold" / "claims.json"
CHROMA_DIR    = "./chroma_langchain_db"
EVAL_DIR      = Path("eval_results")

CLAUSE_COLLECTION = "clausecheck-clauses"
NAIVE_COLLECTION  = "clausecheck-naive"

DENSE_K          = 8      # candidates from Chroma
BM25_K           = 8      # candidates from the keyword index
ENSEMBLE_WEIGHTS = [0.5, 0.5]   # [bm25, dense]
RERANK_TOP_N     = 4      # clauses kept after the cross-encoder

MAX_CHUNK_CHARS    = 1600 # only used when one section is too long to keep whole
CHUNK_OVERLAP      = 150
NAIVE_CHUNK_SIZE   = 1000 # the ablation baseline ignores structure entirely
NAIVE_CHUNK_OVERLAP = 200
NAIVE_K            = 4

MAX_RETRIES_PER_CHECK = 1   # one query rewrite before a check is given up on
STRUCTURED_RETRIES    = 2   # re-prompts allowed when the model breaks format

## 2. Defining Components

### Chat Model

The class labs use `openai/gpt-oss-120b` on Groq and list `llama-3.3-70b-versatile` as a Llama
alternative. Groq has since decommissioned its Llama chat models, so that identifier now returns
a 404 and we use the lab default. The available chat models on the account are checked below.

In [ ]:
model_name = os.getenv("MODEL_NAME", "openai/gpt-oss-120b")
llm = init_chat_model(model_name, model_provider="groq", temperature=0.0)

print("model:", model_name)
print(llm.invoke("Reply with the single word: ready").content)

### Embedding Model

In [ ]:
# Needs Ollama running locally and the model pulled once in a terminal:
#   ollama pull nomic-embed-text

embeddings_model = OllamaEmbeddings(model="nomic-embed-text")

query_embedding = embeddings_model.embed_query("Hello world!")
print("embedding dimension:", len(query_embedding))

### Vector Store

In [ ]:
vector_store_chroma = Chroma(
    collection_name=CLAUSE_COLLECTION,
    embedding_function=embeddings_model,
    persist_directory=CHROMA_DIR,
)

vector_store_naive = Chroma(
    collection_name=NAIVE_COLLECTION,
    embedding_function=embeddings_model,
    persist_directory=CHROMA_DIR,
)

## 3. Indexing

### Loading Documents

`data/policies/manifest.json` maps each PDF to the insurer and product it belongs to. That
metadata is attached to every chunk, and it is what lets us filter retrieval to one policy later.

The three documents here are **generated samples**, written to look like retail health wordings
so the pipeline can be built and tested. They carry a notice saying so on page 1. They differ
from each other in waiting periods, room rent limits, sub-limits and co-payment, which is what
makes the per-policy filter observable. To use real documents, drop the PDFs in, update the
manifest, and re-run this section.

In [ ]:
manifest = json.loads((POLICY_DIR / "manifest.json").read_text(encoding="utf-8"))

for entry in manifest:
    print(f"{entry['policy_id']:26} {entry['insurer']} - {entry['product']}")

In [ ]:
def extract_pages(path):
    """Return [(page_number, text)] with page numbers starting at 1.

    PyPDFLoader gives one Document per page and records a zero-based page
    number in the metadata, so we add one to get the page as printed.
    """
    loader = PyPDFLoader(str(path))
    pages = []
    for position, document in enumerate(loader.load()):
        page_number = int(document.metadata.get("page", position)) + 1
        pages.append((page_number, document.page_content or ""))
    return pages


pages = extract_pages(POLICY_DIR / manifest[0]["file"])
print(f"{len(pages)} pages\n")
print(pages[1][1][:420])

### Splitting Documents: clause-aware chunking

A policy wording is already structured. Section 4.2 is the specific waiting period, 5.1 is the
permanent exclusions, 6.1 is the table of sub-limits. Splitting that every 1000 characters cuts
rules in half and glues unrelated ones together.

So we split on the document's own headings and keep the heading as a prefix on every chunk, so
the provisos that qualify a clause travel with it into the context window. Each chunk also
carries its insurer, clause type, section number and page.

The clause type is assigned by keyword rules rather than by a model: it runs over every chunk of
every document, it has to be repeatable, and the vocabulary of a policy wording is narrow enough
that keywords do the job without a model call.

In [ ]:
# A numbered heading: "4.2 Specific Waiting Periods"
NUMBERED_HEADING = re.compile(r"^\s*(\d+(?:\.\d+)*)[.)]?\s+([A-Za-z][^\n]{2,90})$")

# An unnumbered heading in capitals: "PERMANENT EXCLUSIONS"
CAPS_HEADING = re.compile(r"^\s*([A-Z][A-Z0-9 \-/&,()']{4,80})\s*$")

# Keyword -> clause type. The first hit wins, so the more specific categories
# are listed before the broader ones.
CLAUSE_TYPE_RULES = [
    ("waiting_period", ("waiting period", "waiting periods", "months of continuous",
                        "continuous coverage", "moratorium")),
    ("co_pay",         ("co-pay", "copay", "co pay", "co-payment", "copayment")),
    ("exclusion",      ("exclusion", "excluded", "not payable", "shall not cover",
                        "we will not pay", "not covered")),
    ("room_rent",      ("room rent", "room category", "eligible room", "proportionate")),
    ("sub_limit",      ("sub-limit", "sublimit", "sub limit", "capped at",
                        "limit of", "maximum of", "table of benefits")),
    ("definition",     ("definition", "means ", "shall mean")),
]


def classify_clause(heading, body):
    """Assign a clause type from keywords.

    The heading is tried first and on its own. A clause often mentions other
    categories in passing: the permanent exclusions section says the exclusions
    are "not subject to any waiting period", which would otherwise label the
    whole section as a waiting period.
    """
    for haystack in (heading.lower(), body.lower()):
        for clause_type, keywords in CLAUSE_TYPE_RULES:
            if any(keyword in haystack for keyword in keywords):
                return clause_type
    return "general"

In [ ]:
def flush_chunk(entry, section, heading, lines, page, out):
    """Turn the buffered lines into one or more Documents."""
    body = "\n".join(lines).strip()
    if not body:
        return

    title = f"{section} {heading}".strip()
    metadata = {
        "policy_id":   entry["policy_id"],
        "insurer":     entry["insurer"],
        "product":     entry["product"],
        "section":     section or heading[:40],
        "heading":     heading,
        "clause_type": classify_clause(heading, body),
        "page":        page,
    }

    # The heading is repeated inside the text so it is present in the embedding
    # and in whatever the model finally reads.
    full_text = f"{title}\n{body}" if title else body

    if len(full_text) <= MAX_CHUNK_CHARS:
        out.append(Document(page_content=full_text, metadata=dict(metadata)))
        return

    # A very long section is split further, but every piece keeps the heading.
    splitter = RecursiveCharacterTextSplitter(chunk_size=MAX_CHUNK_CHARS,
                                              chunk_overlap=CHUNK_OVERLAP)
    for piece in splitter.split_text(body):
        out.append(Document(page_content=f"{title}\n{piece}" if title else piece,
                            metadata=dict(metadata)))


def split_into_clauses(pages, entry):
    """Walk the document line by line, starting a new chunk at each heading."""
    documents, buffer = [], []
    section = heading = ""
    chunk_page = 1

    for page_number, text in pages:
        for raw_line in text.splitlines():
            line = raw_line.rstrip()
            if not line.strip():
                continue

            numbered = NUMBERED_HEADING.match(line)
            caps = CAPS_HEADING.match(line) if not numbered else None

            if numbered or caps:
                flush_chunk(entry, section, heading, buffer, chunk_page, documents)
                buffer, chunk_page = [], page_number
                if numbered:
                    section, heading = numbered.group(1), numbered.group(2).strip()
                else:
                    section, heading = "", caps.group(1).strip().title()
            else:
                buffer.append(line.strip())

    flush_chunk(entry, section, heading, buffer, chunk_page, documents)
    return documents

In [ ]:
clause_documents = []
for entry in manifest:
    clause_documents.extend(split_into_clauses(extract_pages(POLICY_DIR / entry["file"]), entry))

print(f"{len(clause_documents)} clause chunks from {len(manifest)} policies\n")

print(f"{'section':9}{'page':6}{'clause_type':17}heading")
print("-" * 70)
for d in clause_documents:
    if d.metadata["policy_id"] == manifest[0]["policy_id"]:
        m = d.metadata
        print(f"{m['section']:9}{m['page']:<6}{m['clause_type']:17}{m['heading'][:32]}")

Section 5.1 is the one to look at. It is the permanent exclusions clause, and its own text says
the exclusions are *"not subject to any waiting period"*. Classifying on the body alone labels it
a waiting period, and the grader then reads an exclusions list when it asked about waiting
periods. Matching the heading first fixes it. This is the kind of bug that produces a confident
wrong answer rather than an error, which is why the clause type is worth getting right.

### The baseline chunking, for the ablation in section 9

In [ ]:
# Fixed-size splitting: no headings, no clause typing, no section metadata.
# This is what section 9 compares the real pipeline against.
naive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=NAIVE_CHUNK_SIZE,
    chunk_overlap=NAIVE_CHUNK_OVERLAP,
    add_start_index=True,
)

naive_documents = []
for entry in manifest:
    for page_number, text in extract_pages(POLICY_DIR / entry["file"]):
        if not text.strip():
            continue
        for piece in naive_splitter.split_text(text):
            naive_documents.append(Document(
                page_content=piece,
                metadata={"policy_id": entry["policy_id"], "insurer": entry["insurer"],
                          "product": entry["product"], "page": page_number, "section": ""},
            ))

print(f"{len(naive_documents)} fixed-size chunks")

### Storing in the Vector Store

In [ ]:
def is_empty(store):
    """True if this collection holds nothing yet."""
    return len(store.get(limit=1)["ids"]) == 0


def stored_count(store):
    return len(store.get()["ids"])


# Each collection is gated separately. Gating both on one count leaves the
# baseline collection empty after a partial run, and the ablation would then
# silently measure the baseline against no context at all.
if is_empty(vector_store_chroma):
    vector_store_chroma.add_documents(
        documents=clause_documents,
        ids=[str(uuid4()) for _ in clause_documents])

if is_empty(vector_store_naive):
    vector_store_naive.add_documents(
        documents=naive_documents,
        ids=[str(uuid4()) for _ in naive_documents])

print("clause collection :", stored_count(vector_store_chroma))
print("naive collection  :", stored_count(vector_store_naive))

## 4. Retrieval

Clause look-up is half semantic and half literal. "Knee replacement" has to match "total or
partial replacement of a knee joint", which needs embeddings. "Clause 4.2" and "48 months" have
to match exactly, which needs keywords. So both indexes are queried and fused with reciprocal
rank fusion, then a cross-encoder re-ranks the shortlist.

**The policy filter is not a refinement.** Retrieving one insurer's waiting period while
adjudicating another insurer's policy gives an answer that is confident and wrong, so the filter
is applied to both halves of the hybrid. BM25 has no metadata filter of its own, so it is built
over only that policy's chunks.

In [ ]:
def build_hybrid(policy_id):
    """BM25 and Chroma over one policy, fused with reciprocal rank fusion."""

    # Dense half: Chroma, filtered to this policy.
    chroma_retriever = vector_store_chroma.as_retriever(
        search_kwargs={"k": DENSE_K, "filter": {"policy_id": policy_id}})

    # Keyword half: built over just this policy's chunks.
    policy_chunks = [d for d in clause_documents if d.metadata["policy_id"] == policy_id]
    bm25_retriever = BM25Retriever.from_documents(documents=policy_chunks)
    bm25_retriever.k = BM25_K

    return EnsembleRetriever(retrievers=[bm25_retriever, chroma_retriever],
                             weights=ENSEMBLE_WEIGHTS)

### Choosing the re-ranker

Lab 5 uses `cross-encoder/qnli-electra-base`. We measured it before adopting it, because a
re-ranker that reorders badly is worse than no re-ranker at all. The probe set is six clause
look-ups where we know which section is the right answer, and the metric is whether that section
survives into the top 4.

In [ ]:
PROBES = {
    'room rent limit per day':                          '3.1',
    'waiting period for total knee replacement surgery':'4.2',
    'is cosmetic surgery permanently excluded':         '5.1',
    'co-payment percentage applicable':                 '7.1',
    'sub-limit for cataract surgery':                   '6.1',
    'pre-existing disease waiting period':              '4.3',
}

probe_policy = manifest[0]["policy_id"]
probe_hybrid = build_hybrid(probe_policy)

candidates = {q: probe_hybrid.invoke(q) for q in PROBES}

results = {"no rerank (fused top 4)": sum(
    gold in [d.metadata["section"] for d in candidates[q][:4]] for q, gold in PROBES.items())}

for label, checkpoint in [("qnli-electra-base (lab 5)", "cross-encoder/qnli-electra-base"),
                          ("ms-marco-MiniLM-L-6-v2",   "cross-encoder/ms-marco-MiniLM-L-6-v2")]:
    rr = CrossEncoderReranker(model=HuggingFaceCrossEncoder(model_name=checkpoint),
                              top_n=RERANK_TOP_N)
    results[label] = sum(
        gold in [d.metadata["section"] for d in rr.compress_documents(candidates[q], q)]
        for q, gold in PROBES.items())

print(f"{'strategy':30}correct clause in top 4")
print("-" * 54)
for label, hits in results.items():
    print(f"{label:30}{hits} / {len(PROBES)}")

`qnli-electra-base` is trained to judge whether a sentence answers a question, not to rank
passages by relevance, and it drops the room rent and co-payment clauses entirely. We keep the
lab's component and API and change only the checkpoint.

Worth being honest about the other half of that table: on a corpus this small the fused
candidates already contain the right clause every time, so re-ranking cannot improve recall here.
Its job is to order the shortlist without losing anything, and that is all we claim for it.

In [ ]:
reranker = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")
reranker_compressor = CrossEncoderReranker(model=reranker, top_n=RERANK_TOP_N)


def build_retriever(policy_id):
    """The full retrieval stack: hybrid fusion with the re-ranker chained on top."""
    return ContextualCompressionRetriever(
        base_compressor=reranker_compressor,
        base_retriever=build_hybrid(policy_id))


retriever = build_retriever(probe_policy)

question = "waiting period for total knee replacement surgery"

print("CLAUSE-AWARE + HYBRID + RERANK")
for d in retriever.invoke(question):
    m = d.metadata
    print(f"   [{m['section']:5}] p{m['page']}  {m['clause_type']:15} {m['heading'][:34]}")

print("\nNAIVE BASELINE (fixed chunks, dense only)")
for d in vector_store_naive.similarity_search(question, k=NAIVE_K,
                                              filter={"policy_id": probe_policy}):
    print("   p%s  %s..." % (d.metadata["page"], d.page_content[:56].strip().replace(chr(10), " ")))

## 5. Structured Output Schemas

Every point where the model hands data back to Python goes through one of these. Nothing
downstream reads raw model text.

In [ ]:
CheckType = Literal["waiting_period", "exclusion", "sub_limit", "room_rent",
                    "co_pay", "sum_insured"]
Decision  = Literal["payable", "not_payable", "partially_payable", "refer_to_human"]


class ParsedClaim(BaseModel):
    """The structured form of a free-text claim scenario."""
    age: Optional[int] = Field(None, description="Age of the insured now, in years")
    age_at_entry: Optional[int] = Field(
        None, description="Age when the policy was FIRST bought, if the scenario states it")
    ailment: str = Field(..., description="The medical condition being claimed for")
    procedure: Optional[str] = Field(None, description="The procedure performed, if named")
    policy_inception_date: Optional[str] = Field(None, description="Policy start, YYYY-MM-DD")
    admission_date: Optional[str] = Field(None, description="Hospital admission, YYYY-MM-DD")
    sum_insured: Optional[float] = Field(None, description="Sum insured in rupees")
    claimed_amount: Optional[float] = Field(None, description="Total amount claimed in rupees")
    room_rent_per_day: Optional[float] = Field(None, description="Actual room rent per day")
    room_category: Optional[str] = Field(None, description="Room category occupied")
    pre_existing: Optional[bool] = Field(None, description="True if stated as pre-existing")


class PlannedCheck(BaseModel):
    """One clause look-up that this particular claim requires."""
    check_id: str = Field(..., description="Short identifier, e.g. wp1")
    check_type: CheckType
    question: str = Field(..., description="Search query to run against the policy wording")
    rationale: str = Field(..., description="Why this claim needs this check")


class ClaimPlan(BaseModel):
    """Output of the parse-and-plan node."""
    claim: ParsedClaim
    checks: List[PlannedCheck]


class Citation(BaseModel):
    """A pointer back into the source document."""
    insurer: str = ""
    product: str = ""
    section: str = ""
    page: int = 0
    quote: str = Field("", description="The sentence the finding rests on")


class CheckEvidence(BaseModel):
    """Grading plus parameter extraction for a single check.

    The grader decides whether the retrieved clauses answer the check, and in
    the same call pulls out whichever number the rules engine will need. One
    model call per check instead of two.
    """
    sufficient: bool = Field(..., description="True only if the clauses answer the question")
    reason: str = Field(..., description="One sentence explaining the grade")
    refined_query: Optional[str] = Field(None, description="Better query, if not sufficient")

    waiting_period_months: Optional[int] = None
    is_excluded: Optional[bool] = Field(None, description="True if explicitly excluded")
    sub_limit_amount: Optional[float] = None
    co_pay_percent: Optional[float] = None
    room_rent_limit_per_day: Optional[float] = Field(None, description="Eligible rent per day")
    room_rent_limit_percent_of_si: Optional[float] = Field(None, description="Rent as % of SI")

    citation: Optional[Citation] = None


class Explanation(BaseModel):
    """The adjudicator's written reasoning.

    The decision is not produced here: it comes from the rules engine. The model
    only explains a decision that has already been made.
    """
    summary: str = Field(..., description="Two or three sentences for the claimant")
    clause_references: List[str] = Field(default_factory=list,
                                         description="Section numbers relied on, e.g. ['4.2']")

### Structured output with a bounded re-prompt

`with_structured_output` is the class idiom, but an open-weight model does not always honour the
schema. When a call comes back unusable we hand the model the error and ask again, up to a fixed
number of attempts, and then raise. A format failure has to surface as an error rather than pass
through as a silently wrong verdict.

In [ ]:
class StructuredOutputError(RuntimeError):
    """The model could not be made to return the requested shape."""


def call_structured(prompt, schema, retries=STRUCTURED_RETRIES):
    """Ask the model for `schema` and return it parsed, or raise."""
    structured_llm = llm.with_structured_output(schema)
    attempt_prompt, last_error = prompt, None

    for _ in range(retries + 1):
        try:
            result = structured_llm.invoke(attempt_prompt)
            return result if isinstance(result, schema) else schema.model_validate(result)
        except Exception as exc:
            last_error = exc
            attempt_prompt = (
                f"{prompt}\n\nYour previous reply was rejected: {exc}\n"
                f"Reply again with values that fit the {schema.__name__} schema exactly. "
                "Do not add commentary.")

    raise StructuredOutputError(
        f"{schema.__name__} could not be produced after {retries + 1} attempts: {last_error}")

## 6. The Benefit Calculator

The model does not decide the claim and does not do the arithmetic. It finds clauses and reads
numbers out of them; this function applies those numbers in the order an insurer would:

1. a permanent exclusion ends the matter
2. an unexpired waiting period ends the matter
3. room rent above the eligible limit reduces the bill proportionately
4. a procedure sub-limit caps what is left
5. the sum insured is the outer bound
6. co-payment is the insured's share of the remainder

Keeping this in Python rather than in the prompt is what makes the output auditable. It also
removes a whole class of error: language models are unreliable at multi-step arithmetic, and no
amount of prompting fixes that.

In [ ]:
DATE_FORMATS = ("%Y-%m-%d", "%d-%m-%Y", "%d/%m/%Y", "%Y/%m/%d", "%d %B %Y", "%B %d, %Y")


def parse_date(value):
    """Accept the handful of formats a model realistically returns."""
    if not value:
        return None
    for fmt in DATE_FORMATS:
        try:
            return datetime.strptime(value.strip(), fmt).date()
        except ValueError:
            continue
    return None


def months_between(start, end):
    """Whole months elapsed, the way a policy counts them."""
    months = (end.year - start.year) * 12 + (end.month - start.month)
    if end.day < start.day:
        months -= 1
    return max(months, 0)


@dataclass
class PolicyParameters:
    """The numbers the grader pulled out of the retrieved clauses."""
    waiting_period_months: Optional[int] = None
    is_excluded: Optional[bool] = None
    sub_limit_amount: Optional[float] = None
    co_pay_percent: Optional[float] = None
    room_rent_limit_per_day: Optional[float] = None
    room_rent_limit_percent_of_si: Optional[float] = None

    def effective_room_rent_limit(self, sum_insured):
        """A room rent cap may be stated in rupees or as a share of the SI.

        If both forms were extracted, the stricter one governs, which is the
        same rule merge_parameters applies everywhere else.
        """
        limits = []
        if self.room_rent_limit_per_day is not None:
            limits.append(self.room_rent_limit_per_day)
        if self.room_rent_limit_percent_of_si is not None and sum_insured:
            limits.append(sum_insured * self.room_rent_limit_percent_of_si / 100.0)
        return min(limits) if limits else None


@dataclass
class Adjudication:
    decision: Decision
    payable_amount: float
    breakdown: List[str] = field(default_factory=list)
    months_elapsed: Optional[int] = None

In [ ]:
def adjudicate(claim, params, unresolved_checks=None):
    """Apply the clauses to the claim, in the order an insurer would."""
    steps = []
    unresolved = unresolved_checks or []

    if unresolved:
        return Adjudication("refer_to_human", 0.0, [
            "Could not establish: " + ", ".join(unresolved),
            "Referred for manual review rather than decided on partial evidence."])

    inception = parse_date(claim.policy_inception_date)
    admission = parse_date(claim.admission_date)
    elapsed = months_between(inception, admission) if inception and admission else None
    if elapsed is not None:
        steps.append(f"Policy in force for {elapsed} months at the date of admission.")

    # 1. A permanent exclusion ends the matter.
    if params.is_excluded:
        steps.append("The condition falls under a permanent exclusion, so nothing is payable.")
        return Adjudication("not_payable", 0.0, steps, elapsed)

    # 2. A waiting period that has not run out also ends the matter. A period of
    #    zero is trivially satisfied, so it does not need the dates: the grader
    #    returns zero when a waiting period exists but this claim is exempt from
    #    it, as an accident is during the initial waiting period.
    if params.waiting_period_months:
        required = params.waiting_period_months
        if elapsed is None:
            steps.append(f"A waiting period of {required} months applies, but the dates given "
                         "are not enough to check it.")
            return Adjudication("refer_to_human", 0.0, steps, elapsed)
        if elapsed < required:
            steps.append(f"The applicable waiting period is {required} months and only "
                         f"{elapsed} months have elapsed, so the claim is not admissible.")
            return Adjudication("not_payable", 0.0, steps, elapsed)
        steps.append(f"The {required} month waiting period is satisfied "
                     f"({elapsed} months elapsed).")

    if claim.claimed_amount is None:
        steps.append("No claimed amount was given, so no figure can be computed.")
        return Adjudication("refer_to_human", 0.0, steps, elapsed)

    admissible = float(claim.claimed_amount)
    steps.append(f"Amount claimed: {admissible:,.2f}")

    # 3. Room rent capping, applied proportionately to the whole bill.
    rent_limit = params.effective_room_rent_limit(claim.sum_insured)
    actual_rent = claim.room_rent_per_day
    # A limit of zero is not a real cap; it means the grader found nothing.
    if rent_limit and actual_rent and actual_rent > rent_limit:
        factor = rent_limit / actual_rent
        admissible *= factor
        steps.append(f"Room rent of {actual_rent:,.2f} per day exceeds the eligible "
                     f"{rent_limit:,.2f}. Associated charges reduced proportionately "
                     f"({factor:.1%}) to {admissible:,.2f}.")

    # 4. A procedure sub-limit caps what is left.
    if params.sub_limit_amount is not None and admissible > params.sub_limit_amount:
        admissible = float(params.sub_limit_amount)
        steps.append(f"Procedure sub-limit applied, capping the amount at {admissible:,.2f}.")

    # 5. The sum insured is the outer bound.
    if claim.sum_insured is not None and admissible > claim.sum_insured:
        admissible = float(claim.sum_insured)
        steps.append(f"Capped at the sum insured of {admissible:,.2f}.")

    # 6. Co-payment is the insured's own share of what remains.
    payable = admissible
    if params.co_pay_percent:
        payable = admissible * (1 - params.co_pay_percent / 100.0)
        steps.append(f"Co-payment of {params.co_pay_percent:.0f}% deducted, "
                     f"leaving {payable:,.2f}.")

    # Never report a negative figure. A co-payment above 100%, or a negative
    # amount extracted from a clause, would otherwise surface as a negative
    # payout in the app.
    payable = max(0.0, round(payable, 2))
    steps.append(f"Net payable: {payable:,.2f}")

    # "Nothing was deducted" is tested before "nothing is payable", so a claim
    # for zero is reported as paid in full rather than as denied.
    if abs(payable - float(claim.claimed_amount)) < 1.0:
        decision = "payable"
    elif payable <= 0:
        decision = "not_payable"
    else:
        decision = "partially_payable"

    return Adjudication(decision, payable, steps, elapsed)

### Checking the calculator

The rules engine is plain Python, so it can be tested without an API key or a vector store. These
assertions mirror cases in the gold set: if a gold amount and this engine disagree, one of the two
is wrong and we want to know before any model is involved.

In [ ]:
def demo_claim(**kwargs):
    base = dict(ailment="test condition", claimed_amount=100000.0)
    base.update(kwargs)
    return ParsedClaim(**base)

# Months are counted the way a policy counts them, not as calendar differences.
assert months_between(parse_date("2022-01-10"), parse_date("2025-03-20")) == 38
assert months_between(parse_date("2024-06-01"), parse_date("2025-05-31")) == 11

# Gold case c01: a 24 month waiting period, 12 months elapsed.
r = adjudicate(demo_claim(policy_inception_date="2024-06-01", admission_date="2025-06-15",
                          sum_insured=500000.0, claimed_amount=320000.0, room_rent_per_day=4000.0),
               PolicyParameters(waiting_period_months=24, is_excluded=False))
assert r.decision == "not_payable" and r.payable_amount == 0.0

# Gold case c02: waiting satisfied, the 1,50,000 knee sub-limit binds.
r = adjudicate(demo_claim(policy_inception_date="2022-01-10", admission_date="2025-03-20",
                          sum_insured=500000.0, claimed_amount=300000.0, room_rent_per_day=4000.0),
               PolicyParameters(waiting_period_months=24, is_excluded=False,
                                sub_limit_amount=150000.0, room_rent_limit_percent_of_si=1.0))
assert r.decision == "partially_payable" and r.payable_amount == 150000.0

# Gold case c06: 5,000 eligible against 8,000 charged is a 62.5% factor.
r = adjudicate(demo_claim(policy_inception_date="2023-01-01", admission_date="2025-02-10",
                          sum_insured=500000.0, claimed_amount=200000.0, room_rent_per_day=8000.0),
               PolicyParameters(waiting_period_months=0, is_excluded=False,
                                room_rent_limit_per_day=5000.0))
assert r.payable_amount == 125000.0

# Gold case c14: room rent, then sub-limit, then co-payment, in that order.
r = adjudicate(demo_claim(policy_inception_date="2020-01-01", admission_date="2025-04-01",
                          sum_insured=500000.0, claimed_amount=400000.0, room_rent_per_day=8000.0),
               PolicyParameters(waiting_period_months=24, is_excluded=False,
                                sub_limit_amount=150000.0, co_pay_percent=20.0,
                                room_rent_limit_percent_of_si=1.0))
assert r.payable_amount == 120000.0

# Gold case c19: a 5,00,000 bill against a 3,00,000 sum insured.
r = adjudicate(demo_claim(policy_inception_date="2020-03-01", admission_date="2025-01-05",
                          sum_insured=300000.0, claimed_amount=500000.0, room_rent_per_day=5000.0),
               PolicyParameters(waiting_period_months=0, is_excluded=False,
                                room_rent_limit_percent_of_si=2.0))
assert r.decision == "partially_payable" and r.payable_amount == 300000.0

# An unresolved check must never be decided on partial evidence.
r = adjudicate(demo_claim(policy_inception_date="2023-01-01", admission_date="2025-02-01"),
               PolicyParameters(), unresolved_checks=["exclusion (ex1)"])
assert r.decision == "refer_to_human"

# --- guards against bad values coming back from the grader ---

# A co-payment above 100% must not produce a negative payout.
r = adjudicate(demo_claim(claimed_amount=100000.0),
               PolicyParameters(is_excluded=False, co_pay_percent=120.0))
assert r.payable_amount == 0.0 and r.decision == "not_payable"

# A zero-month waiting period is satisfied without needing the dates. This is
# how an accident inside the initial waiting period is encoded (gold case c15).
r = adjudicate(demo_claim(claimed_amount=80000.0),
               PolicyParameters(waiting_period_months=0, is_excluded=False))
assert r.decision == "payable" and r.payable_amount == 80000.0

# A room rent limit of zero means "nothing found", not "everything is deducted".
r = adjudicate(demo_claim(claimed_amount=200000.0, room_rent_per_day=8000.0),
               PolicyParameters(is_excluded=False, room_rent_limit_per_day=0.0))
assert r.payable_amount == 200000.0

# Where both forms of a room rent cap were extracted, the stricter one governs.
assert PolicyParameters(room_rent_limit_per_day=5000.0,
                        room_rent_limit_percent_of_si=0.5
                        ).effective_room_rent_limit(300000) == 1500.0

print("all calculator checks passed")

In [ ]:
# A worked example, printed step by step: the 64-year-old's knee replacement.
claim = ParsedClaim(ailment="osteoarthritis", procedure="total knee replacement", age=64,
                    policy_inception_date="2020-01-01", admission_date="2025-04-01",
                    sum_insured=500000, claimed_amount=400000, room_rent_per_day=8000)
params = PolicyParameters(waiting_period_months=24, is_excluded=False, sub_limit_amount=150000,
                          co_pay_percent=20, room_rent_limit_percent_of_si=1.0)

result = adjudicate(claim, params)
print("decision:", result.decision, "| payable:", f"{result.payable_amount:,.2f}\n")
for step in result.breakdown:
    print(" -", step)

## 7. The Agent

Five nodes and one conditional edge:

```
  parse_and_plan -> retrieve -> grade -+-> calculate -> adjudicate -> END
                        ^              |
                        +--- retry ----+
```

The edge out of `grade` is what makes this a graph rather than a chain. If the retrieved clauses
do not answer a check, that check goes round again with a query the model rewrites. A check that
fails twice is left unresolved, and any unresolved check forces the whole claim to
`refer_to_human` instead of a guess.

Each node returns only the keys it changed, as in the class lab.

### State

In [ ]:
class ClaimState(TypedDict, total=False):
    claim_text: str
    policy_id: str
    claim: Optional[ParsedClaim]
    checks: List[dict]
    params: PolicyParameters
    adjudication: Adjudication
    explanation: Optional[Explanation]
    trace: List[str]


def format_clauses(documents):
    """Lay the retrieved clauses out for the prompt, with their provenance."""
    blocks = []
    for position, doc in enumerate(documents, start=1):
        m = doc.metadata
        blocks.append(f"[{position}] Section {m.get('section', '?')} "
                      f"(page {m.get('page', '?')}, type {m.get('clause_type', 'general')})\n"
                      f"{doc.page_content}")
    return "\n\n".join(blocks) if blocks else "(nothing retrieved)"

### Node 1: parse and plan

In [ ]:
PLAN_PROMPT = """Read this health insurance claim scenario and do two things.

First, extract the facts into the claim object. Use null for anything the
scenario does not state. Dates must be YYYY-MM-DD.

`age` is the age now. `age_at_entry` is the age when the policy was first
bought, and you must fill it in whenever the scenario says so, for example
"he bought the policy at the age of 62". Some co-payment clauses turn on the
age at entry rather than the age now, so this cannot be dropped.

Second, list the clause checks this claim needs. Only include a check the
scenario actually calls for:
  waiting_period - the condition may be subject to a waiting period
  exclusion      - the condition may be permanently excluded
  sub_limit      - the procedure may carry a monetary cap
  room_rent      - a room rent limit may apply (include if a room rent is given)
  co_pay         - a co-payment may apply, for example for an older insured
  sum_insured    - the bill may exceed the sum insured, so the cap should be cited

Each check needs a `question` written as a search query against a policy
wording, not as a question to a person. For example:
"waiting period for joint replacement surgery".

CLAIM SCENARIO:
{claim_text}"""


def parse_and_plan(state: ClaimState):
    plan = call_structured(PLAN_PROMPT.format(claim_text=state["claim_text"]), ClaimPlan)

    checks = [{"check_id": c.check_id, "check_type": c.check_type, "question": c.question,
               "query": c.question, "rationale": c.rationale, "attempts": 0,
               "status": "pending", "documents": [], "seen_documents": [], "evidence": None}
              for c in plan.checks]

    trace = [f"Parsed claim for '{plan.claim.ailment}' and planned {len(checks)} check(s): "
             + ", ".join(c["check_type"] for c in checks)]

    # An exclusion check is never optional. Every other parameter defaults to
    # "no restriction found", which is the safe direction; is_excluded defaulting
    # to None means "not excluded", which pays a claim the policy may forbid. If
    # the planner did not ask, we ask anyway.
    if not any(c["check_type"] == "exclusion" for c in checks):
        subject = plan.claim.procedure or plan.claim.ailment
        checks.append({"check_id": "ex_auto", "check_type": "exclusion",
                       "question": f"is {subject} a permanent exclusion under this policy",
                       "query": f"permanent exclusions {subject}",
                       "rationale": "Added automatically: every claim is checked for exclusions.",
                       "attempts": 0, "status": "pending",
                       "documents": [], "seen_documents": [], "evidence": None})
        trace.append("Added a mandatory exclusion check that the plan omitted.")

    return {"claim": plan.claim, "checks": checks, "trace": trace}

### Node 2: retrieve

In [ ]:
def retrieve(state: ClaimState):
    policy_retriever = build_retriever(state["policy_id"])
    checks = state["checks"]
    trace = list(state.get("trace", []))

    for check in checks:
        if check["status"] != "pending":
            continue
        documents = policy_retriever.invoke(check["query"])
        check["documents"] = documents
        # Keep every clause the check ever saw, not just the last pass. The
        # recall metric asks whether retrieval found the governing clause at
        # all; a retry that wanders off should not erase a pass that found it.
        check["seen_documents"] = check.get("seen_documents", []) + documents
        trace.append(f'[{check["check_id"]}] retrieved {len(documents)} clause(s) '
                     f'for "{check["query"]}"')

    return {"checks": checks, "trace": trace}

### Node 3: grade, and extract the parameters

In [ ]:
GRADE_PROMPT = """You are checking one point against a policy wording.

CHECK TYPE: {check_type}
QUESTION: {question}

CLAIM FACTS:
{claim_facts}

RETRIEVED CLAUSES:
{clauses}

Decide whether these clauses actually answer the question.

If they do, set sufficient to true and fill in only the field that belongs to
this check type:
  waiting_period -> waiting_period_months, the period that applies to THIS
                    condition. If the condition is pre-existing, use the
                    pre-existing disease waiting period. Return 0 if a waiting
                    period exists but this claim is exempt from it, for example
                    an accident during the initial waiting period, or if the
                    period is stated in days and has already elapsed.
  exclusion      -> is_excluded. True only if this condition is named as a
                    permanent exclusion. A condition that merely carries a
                    waiting period is NOT excluded, even where the waiting
                    period clause uses the word "excluded".
  sub_limit      -> sub_limit_amount, in rupees, for THIS procedure only.
                    Ignore caps on other benefits listed in the same table.
  room_rent      -> room_rent_limit_per_day in rupees, or
                    room_rent_limit_percent_of_si if stated as a percentage.
  co_pay         -> co_pay_percent, only if it applies to this claimant. Read
                    the condition carefully: a clause on the age AT ENTRY is
                    tested against age_at_entry, not the age now.
  sum_insured    -> no value to extract; just cite the clause that caps the
                    company's liability at the sum insured.

Fill in ONLY the field named above for this check type. Leave every other
numeric field null, even if the clauses happen to mention it.

Also fill in `citation` with the section, page and the sentence you relied on.

If the clauses do not answer the question, set sufficient to false and put a
better search query in refined_query. Never invent a number that is not written
in the clauses."""


def grade(state: ClaimState):
    checks = state["checks"]
    trace = list(state.get("trace", []))
    claim_facts = (state["claim"].model_dump_json(indent=1, exclude_none=True)
                   if state.get("claim") else "(none)")

    for check in checks:
        if check["status"] != "pending":
            continue

        try:
            evidence = call_structured(
                GRADE_PROMPT.format(check_type=check["check_type"], question=check["question"],
                                    claim_facts=claim_facts,
                                    clauses=format_clauses(check["documents"])),
                CheckEvidence)
        except StructuredOutputError as exc:
            check["status"] = "unresolved"
            check["error"] = str(exc)
            trace.append(f'[{check["check_id"]}] grader output unusable, giving up')
            continue

        check["evidence"] = evidence

        if evidence.sufficient:
            check["status"] = "resolved"
            trace.append(f'[{check["check_id"]}] evidence accepted: {evidence.reason}')
        elif check["attempts"] < MAX_RETRIES_PER_CHECK:
            check["attempts"] += 1
            check["query"] = evidence.refined_query or check["question"]
            trace.append(f'[{check["check_id"]}] evidence weak ({evidence.reason}); '
                         f'retrying with "{check["query"]}"')
        else:
            check["status"] = "unresolved"
            trace.append(f'[{check["check_id"]}] still unresolved after a retry, '
                         'will refer to human')

    return {"checks": checks, "trace": trace}


def needs_another_pass(state: ClaimState):
    """Conditional edge: go round again only if some check is still pending."""
    if any(check["status"] == "pending" for check in state["checks"]):
        return "retrieve"
    return "calculate"

### Node 4: calculate

In [ ]:
# Which parameter a check is allowed to contribute. CheckEvidence carries all
# six fields on every response, so without this guard a grader answering the
# exclusion check could also return, say, a sub-limit, and the merge below would
# let it govern the whole claim. That is not hypothetical: section 4.2 of the
# sample wording reads "shall be excluded until the expiry of 24 months", so an
# exclusion grader shown that clause can plausibly return is_excluded=True and
# deny a claim that is merely inside a waiting period.
FIELDS_OWNED_BY_CHECK = {
    "waiting_period": {"waiting_period_months"},
    "exclusion":      {"is_excluded"},
    "sub_limit":      {"sub_limit_amount"},
    "room_rent":      {"room_rent_limit_per_day", "room_rent_limit_percent_of_si"},
    "co_pay":         {"co_pay_percent"},
    "sum_insured":    set(),   # citation only; the cap comes from the schedule
}


def merge_parameters(checks):
    """Combine what each check found, taking the most restrictive reading.

    Where two clauses both apply, the longer waiting period, the lower sub-limit
    and the higher co-payment govern. A check only contributes the parameter it
    was sent to find.
    """
    waiting, sub_limits, co_pays, rent_amounts, rent_percents = [], [], [], [], []
    excluded = None

    for check in checks:
        evidence = check.get("evidence")
        if evidence is None or check["status"] != "resolved":
            continue

        owned = FIELDS_OWNED_BY_CHECK.get(check["check_type"], set())

        def kept(field_name, value):
            """Ignore a value this check had no business extracting."""
            return value if field_name in owned else None

        evidence = evidence.model_copy(update={
            name: kept(name, getattr(evidence, name))
            for name in ("waiting_period_months", "is_excluded", "sub_limit_amount",
                         "co_pay_percent", "room_rent_limit_per_day",
                         "room_rent_limit_percent_of_si")
        })

        if evidence.waiting_period_months is not None:
            waiting.append(evidence.waiting_period_months)
        if evidence.sub_limit_amount is not None:
            sub_limits.append(evidence.sub_limit_amount)
        if evidence.co_pay_percent is not None:
            co_pays.append(evidence.co_pay_percent)
        if evidence.room_rent_limit_per_day is not None:
            rent_amounts.append(evidence.room_rent_limit_per_day)
        if evidence.room_rent_limit_percent_of_si is not None:
            rent_percents.append(evidence.room_rent_limit_percent_of_si)
        if evidence.is_excluded is True:
            excluded = True
        elif evidence.is_excluded is False and excluded is None:
            excluded = False

    return PolicyParameters(
        waiting_period_months=max(waiting) if waiting else None,
        is_excluded=excluded,
        sub_limit_amount=min(sub_limits) if sub_limits else None,
        co_pay_percent=max(co_pays) if co_pays else None,
        room_rent_limit_per_day=min(rent_amounts) if rent_amounts else None,
        room_rent_limit_percent_of_si=min(rent_percents) if rent_percents else None)


def calculate(state: ClaimState):
    params = merge_parameters(state["checks"])
    unresolved = [f'{c["check_type"]} ({c["check_id"]})'
                  for c in state["checks"] if c["status"] == "unresolved"]

    result = adjudicate(state["claim"], params, unresolved)
    trace = list(state.get("trace", []))
    trace.append(f"Rules engine decision: {result.decision} ({result.payable_amount:,.2f})")
    return {"params": params, "adjudication": result, "trace": trace}

In [ ]:
# The type guard is the difference between a correct verdict and a confident
# wrong one, so it gets its own check. An exclusion grader that also volunteers
# a 3,000 ambulance sub-limit must not cap a 1,50,000 knee replacement.
leaky_checks = [
    {"check_type": "exclusion", "status": "resolved",
     "evidence": CheckEvidence(sufficient=True, reason="", is_excluded=False,
                               sub_limit_amount=3000.0, waiting_period_months=99)},
    {"check_type": "sub_limit", "status": "resolved",
     "evidence": CheckEvidence(sufficient=True, reason="", sub_limit_amount=150000.0)},
]
merged = merge_parameters(leaky_checks)
assert merged.sub_limit_amount == 150000.0, merged
assert merged.waiting_period_months is None, merged
assert merged.is_excluded is False, merged

print("parameter merge respects check ownership")

### Node 5: adjudicate, which only explains

In [ ]:
EXPLAIN_PROMPT = """Write the explanation that goes back to the claimant.

The decision has already been made by the rules engine. Do not change it,
soften it, or add conditions to it. Explain it.

DECISION: {decision}
AMOUNT PAYABLE: {amount}

HOW IT WAS REACHED:
{breakdown}

CLAUSES RELIED ON:
{citations}

Write two or three plain sentences. Name the clause that decided the outcome.
List the section numbers you referred to in clause_references."""


def adjudicate_node(state: ClaimState):
    result = state["adjudication"]
    trace = list(state.get("trace", []))

    citations = []
    for check in state["checks"]:
        evidence = check.get("evidence")
        if evidence and evidence.citation:
            c = evidence.citation
            citations.append(f"- Section {c.section} (page {c.page}): {c.quote}")

    try:
        explanation = call_structured(
            EXPLAIN_PROMPT.format(decision=result.decision,
                                  amount=f"{result.payable_amount:,.2f}",
                                  breakdown="\n".join(f"- {s}" for s in result.breakdown),
                                  citations="\n".join(citations) if citations else "(none)"),
            Explanation)
        trace.append("Explanation written.")
    except StructuredOutputError:
        # The decision still stands; only the prose is missing.
        explanation = None
        trace.append("Explanation could not be generated; the decision is unaffected.")

    return {"explanation": explanation, "trace": trace}

### Building the graph

In [ ]:
graph_builder = StateGraph(ClaimState)

graph_builder.add_node("parse_and_plan", parse_and_plan)
graph_builder.add_node("retrieve", retrieve)
graph_builder.add_node("grade", grade)
graph_builder.add_node("calculate", calculate)
graph_builder.add_node("adjudicate", adjudicate_node)

graph_builder.add_edge(START, "parse_and_plan")
graph_builder.add_edge("parse_and_plan", "retrieve")
graph_builder.add_edge("retrieve", "grade")
graph_builder.add_conditional_edges("grade", needs_another_pass,
                                    {"retrieve": "retrieve", "calculate": "calculate"})
graph_builder.add_edge("calculate", "adjudicate")
graph_builder.add_edge("adjudicate", END)

graph = graph_builder.compile()

In [ ]:
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # draw_mermaid_png calls mermaid.ink, so it needs a network connection.
    print(graph.get_graph().draw_ascii())

### Running a claim

In [ ]:
# The 20 labelled scenarios. Loaded here because the app in section 8 and the
# evaluation in section 9 both use them, not only the demo below.
gold = json.loads(GOLD_PATH.read_text(encoding="utf-8"))
print(f"{len(gold)} gold scenarios")

In [ ]:
case = next(c for c in gold if c["id"] == "c14")

print("SCENARIO:", case["scenario"])
print("EXPECTED:", case["expected_decision"], f'{case["expected_amount"]:,}')
print("-" * 72)

state = graph.invoke({"claim_text": case["scenario"], "policy_id": case["policy_id"],
                      "trace": []})

for line in state["trace"]:
    print(" ", line)

In [ ]:
adj = state["adjudication"]
print("DECISION :", adj.decision)
print("PAYABLE  :", f"{adj.payable_amount:,.2f}\n")
for step in adj.breakdown:
    print(" -", step)

if state.get("explanation"):
    print("\nEXPLANATION:", state["explanation"].summary)
    print("CLAUSES    :", ", ".join(state["explanation"].clause_references))

### Watching the abstain path

The referral route is easier to trust once you have seen it fire. Asking about a benefit the
wording does not contain should end in a referral, not an invented number.

In [ ]:
state_gap = graph.invoke({
    "claim_text": ("A 40 year old man claims 60,000 for an outpatient physiotherapy course of "
                   "20 sessions after a shoulder injury. Policy in force since 2021-01-01, "
                   "treatment on 2025-02-10. Sum insured 3,00,000."),
    "policy_id": manifest[0]["policy_id"],
    "trace": []})

for line in state_gap["trace"]:
    print(" ", line)
print("\nDECISION:", state_gap["adjudication"].decision)

## 8. The Application

Pick a policy, paste a claim, and the agent runs. The trace panel shows the path taken through
the graph, including any retry, which is what makes the agent visible during the demo.

In [ ]:
policy_choices = [f'{e["insurer"]} - {e["product"]}' for e in manifest]
policy_lookup = {f'{e["insurer"]} - {e["product"]}': e["policy_id"] for e in manifest}
policy_label_for = {e["policy_id"]: f'{e["insurer"]} - {e["product"]}' for e in manifest}

# Each example carries the policy it belongs to. Loading a scenario without
# switching the policy would adjudicate, say, a TotalCare claim against
# SecureHealth's sub-limits, which is the exact cross-policy error the metadata
# filter exists to prevent.
example_scenarios = {f'{c["id"]}: {c["tests"]}': (c["scenario"], c["policy_id"]) for c in gold}


def adjudicate_claim(scenario, policy_label):
    """What the Gradio button calls."""
    if not scenario.strip():
        return "", "", "", ""
    if policy_label not in policy_lookup:
        return "", "Select a policy first.", "", ""

    try:
        result_state = graph.invoke({"claim_text": scenario,
                                     "policy_id": policy_lookup[policy_label],
                                     "trace": []})
    except Exception as exc:
        # A StructuredOutputError is raised by design when the model will not
        # produce a usable plan. Surface it instead of leaving stale output.
        return ("REFER TO HUMAN",
                f"The claim could not be processed automatically: {exc}", "", "")

    adjudication = result_state["adjudication"]

    verdict = f'{adjudication.decision.replace("_", " ").upper()}   '
    verdict += f'Rs {adjudication.payable_amount:,.2f}'

    explanation = (result_state["explanation"].summary
                   if result_state.get("explanation") else
                   "Evidence was not sufficient to decide this claim; it has been referred.")

    working = "\n".join(f"- {s}" for s in adjudication.breakdown)

    clauses = []
    for check in result_state["checks"]:
        evidence = check.get("evidence")
        if evidence and evidence.citation and evidence.citation.section:
            clauses.append(f'[{check["check_type"]}] Section {evidence.citation.section} '
                           f'(page {evidence.citation.page}): {evidence.citation.quote}')
    clause_text = "\n\n".join(clauses) if clauses else "(no clause was cited)"

    trace_text = "\n".join(result_state["trace"])

    return verdict, explanation, working + "\n\n--- CLAUSES ---\n\n" + clause_text, trace_text


def load_example(name):
    """Load the scenario AND switch to the policy it belongs to."""
    scenario, policy_id = example_scenarios.get(name, ("", None))
    return scenario, policy_label_for.get(policy_id, policy_choices[0])

In [ ]:
# Gradio Blocks layout
with gr.Blocks(title="ClauseCheck") as demo:
    gr.Markdown("## ClauseCheck")
    gr.Markdown("Health insurance claim adjudication, grounded in the policy wording. "
                "Every figure traces back to a clause.")

    with gr.Row():
        policy_dropdown = gr.Dropdown(label="Adjudicate against",
                                      choices=policy_choices, value=policy_choices[0])
        example_dropdown = gr.Dropdown(label="Load an example scenario",
                                       choices=list(example_scenarios), value=None)

    claim_input = gr.Textbox(
        label="Claim scenario", lines=4,
        placeholder=("A 45 year old woman underwent a total knee replacement. Her policy has "
                     "been in force since 2022-01-10 and she was admitted on 2025-03-20. Sum "
                     "insured is 5,00,000. The hospital bill is 3,00,000. Room rent was 4,000 "
                     "per day."))

    run_button = gr.Button("Adjudicate", variant="primary")

    verdict_output = gr.Textbox(label="Decision")
    explanation_output = gr.Textbox(label="Explanation", lines=3)

    with gr.Accordion("How the amount was reached, and the clauses relied on", open=True):
        working_output = gr.Textbox(label="", lines=14)

    with gr.Accordion("Agent trace", open=False):
        trace_output = gr.Textbox(label="", lines=10)

    example_dropdown.change(fn=load_example, inputs=[example_dropdown],
                            outputs=[claim_input, policy_dropdown])
    run_button.click(fn=adjudicate_claim,
                     inputs=[claim_input, policy_dropdown],
                     outputs=[verdict_output, explanation_output, working_output, trace_output])

In [ ]:
# In a VSCode notebook the inline iframe often stays blank, so serve the app on
# localhost and open it in the browser. launch() returns straight away.
demo.launch(inline=False, inbrowser=True, share=False)
print("If the browser did not open, go to the URL printed above (normally http://127.0.0.1:7860).")

## 9. Evaluation and Ablation

RAGAS is not part of the class stack, and its `datasets` dependency does not install on every
machine, so the metrics are built from the same components as the rest of the notebook. Because
the gold set records the governing clause for each of its 20 scenarios, all but one of these
metrics are deterministic and need no model at all.

| Metric | Meaning |
|---|---|
| Verdict accuracy | the decision matches the label |
| Amount accuracy | the rupee figure matches, within one rupee |
| Citation accuracy | the governing section was actually cited |
| Context recall | retrieval surfaced the governing clause |
| Faithfulness | share of explanation statements the clauses support (LLM as judge) |
| Abstention rate | share referred to a human instead of decided |

Abstention is reported separately because it is the one number that looks good when it is bad.
Nine of the twenty scenarios expect a payable amount of zero, and a referral also carries zero,
so an abstention must not be allowed to count as a correct amount: a pipeline that refuses every
claim would otherwise score 45% on amount accuracy while getting every verdict wrong.

The baseline is naive RAG: fixed-size chunks, dense retrieval only, no clause typing, no
re-ranking, no planning, no rules engine. One retrieval and one model call asked for the verdict
directly.

**A note on cost, because it shapes what we can report.** Groq's free tier allows 200,000 tokens
per day per model. The agent spends roughly six calls per case and each grading call carries four
clauses, so the full 20-case ablation costs about 264,000 tokens and cannot finish in one day on
one model. The cell below therefore defaults to eight cases with the judge off, which fits inside
a day's budget. The limit is applied per model, so running the baseline on `openai/gpt-oss-20b`
and the agent on `openai/gpt-oss-120b` is another way to fit the whole set in. If a run is cut
short by the quota it stops cleanly, keeps the cases that completed, and labels the table
provisional rather than reporting a partial run as a finished one.

### The naive baseline

In [ ]:
class NaiveVerdict(BaseModel):
    """What the baseline is asked to produce in one shot."""
    decision: Decision = Field(..., description="payable, not_payable or partially_payable")
    payable_amount: float = Field(..., description="Amount payable in rupees, 0 if none")
    clause_reference: Optional[str] = Field(None, description="Section number relied on")


NAIVE_TEMPLATE = """You are an insurance claims analyst. Using only the policy
extracts below, decide whether the claim is payable and for how much.

POLICY EXTRACTS:
{context}

CLAIM:
{claim}

Give the decision, the rupee amount payable, and the clause you relied on."""


def run_naive(scenario, policy_id):
    documents = vector_store_naive.similarity_search(scenario, k=NAIVE_K,
                                                     filter={"policy_id": policy_id})
    context = "\n\n".join(d.page_content for d in documents)

    prompt = ChatPromptTemplate.from_template(NAIVE_TEMPLATE)
    chain = prompt | llm.with_structured_output(NaiveVerdict)
    verdict = chain.invoke({"context": context, "claim": scenario})

    # Fixed-size chunks carry no section metadata, so the section numbers have
    # to be read back out of the text. That is itself part of the finding.
    sections = []
    for d in documents:
        sections.extend(re.findall(r"\b(\d+\.\d+)\b", d.page_content))

    return verdict, sections

### The faithfulness judge

In [ ]:
class FaithfulnessScore(BaseModel):
    supported: int = Field(..., description="How many statements the clauses support")
    total: int = Field(..., description="How many statements the explanation makes")


FAITHFULNESS_TEMPLATE = """Break the explanation into individual factual statements.
For each one, decide whether the clauses below support it. A statement is
supported only if the clauses actually say it. Do not use your own knowledge of
insurance.

CLAUSES:
{clauses}

EXPLANATION:
{explanation}

Report how many statements there are in total and how many are supported."""


def score_faithfulness(explanation, clauses):
    """Fraction of the explanation that the retrieved clauses actually support."""
    if not explanation.strip():
        return None
    prompt = ChatPromptTemplate.from_template(FAITHFULNESS_TEMPLATE)
    chain = prompt | llm.with_structured_output(FaithfulnessScore)
    try:
        result = chain.invoke({"clauses": clauses, "explanation": explanation})
    except Exception:
        return None
    if not result.total:
        return None
    # A judge that reports more supported statements than total would otherwise
    # give a faithfulness above 100%.
    return max(0.0, min(1.0, result.supported / result.total))

### Running both pipelines over the gold set

In [ ]:
def evaluate_case_full(case, judge=True):
    started = time.time()
    s = graph.invoke({"claim_text": case["scenario"], "policy_id": case["policy_id"],
                      "trace": []})
    adjudication = s["adjudication"]

    cited, retrieved, clause_text = [], [], []
    for check in s.get("checks", []):
        for d in check.get("seen_documents", check.get("documents", [])):
            if d.metadata.get("section"):
                retrieved.append(str(d.metadata["section"]))
            clause_text.append(d.page_content)
        evidence = check.get("evidence")
        if evidence and evidence.citation and evidence.citation.section:
            cited.append(str(evidence.citation.section))

    # Only the grounded per-check citations count, not the sections the
    # explanation happens to mention. The baseline gets exactly one citation,
    # so counting both here would score the two pipelines on different things.
    explanation = s.get("explanation")

    faithfulness = (score_faithfulness(explanation.summary, "\n\n".join(clause_text[:12]))
                    if judge and explanation else None)

    return {"case_id": case["id"], "decision": adjudication.decision,
            "amount": adjudication.payable_amount, "cited": cited,
            "retrieved": retrieved, "faithfulness": faithfulness,
            "seconds": time.time() - started}


def evaluate_case_naive(case, judge=False):
    started = time.time()
    verdict, sections = run_naive(case["scenario"], case["policy_id"])
    return {"case_id": case["id"], "decision": verdict.decision,
            "amount": verdict.payable_amount,
            "cited": [verdict.clause_reference] if verdict.clause_reference else [],
            "retrieved": sections, "faithfulness": None,
            "seconds": time.time() - started}


def amount_is_correct(row, case):
    """An abstention is never a correct amount.

    refer_to_human carries payable_amount = 0, and 9 of the 20 gold cases
    expect 0, so counting abstentions as matches would let a pipeline that
    refuses to decide anything score 45% here.
    """
    if row["decision"] == "refer_to_human":
        return False
    return abs(row["amount"] - case["expected_amount"]) <= 1


def summarise(rows, cases):
    by_id = {c["id"]: c for c in cases}
    rows = [r for r in rows if r["case_id"] in by_id]
    n = len(rows)
    if n == 0:
        return {}

    scored = [r["faithfulness"] for r in rows if r["faithfulness"] is not None]
    return {
        "verdict_accuracy":   sum(r["decision"] == by_id[r["case_id"]]["expected_decision"]
                                  for r in rows) / n,
        "amount_accuracy":    sum(amount_is_correct(r, by_id[r["case_id"]])
                                  for r in rows) / n,
        "citation_accuracy":  sum(by_id[r["case_id"]]["governing_section"] in r["cited"]
                                  for r in rows) / n,
        "context_recall":     sum(by_id[r["case_id"]]["governing_section"] in r["retrieved"]
                                  for r in rows) / n,
        "faithfulness":       (sum(scored) / len(scored)) if scored else None,
        "abstention_rate":    sum(r["decision"] == "refer_to_human" for r in rows) / n,
        "mean_seconds":       sum(r["seconds"] for r in rows) / n,
    }

In [ ]:
# Budget. Groq's free tier allows 200,000 tokens per day per model, and the
# full 20-case ablation costs roughly 264,000: about 200k for the agent (six
# calls a case, each carrying four clauses), 24k for the baseline and 40k for
# the judge. It therefore cannot finish in one day on one model at full size.
#
#   EVAL_CASES = 8, JUDGE = False   ~ 90k tokens, fits comfortably
#   EVAL_CASES = None, JUDGE = False ~ 224k, needs a second day or a second model
#
# The two pipelines can also be pointed at different models, since the daily
# limit is applied per model.
RUN_EVAL = True
EVAL_CASES = 8        # None for all 20
JUDGE = False         # the faithfulness judge costs one extra call per case

EVAL_DIR.mkdir(exist_ok=True)
ablation_path = EVAL_DIR / "ablation.json"

if not RUN_EVAL and not ablation_path.exists():
    raise FileNotFoundError("No saved results to load. Set RUN_EVAL = True and re-run this cell.")

if RUN_EVAL:
    eval_cases = gold[:EVAL_CASES] if EVAL_CASES else gold
    naive_rows, full_rows = [], []

    def run_pipeline(label, runner, rows):
        """Run one pipeline, and stop cleanly if the token budget runs out."""
        print(f"{label} over {len(eval_cases)} cases")
        for case in eval_cases:
            try:
                row = runner(case, judge=JUDGE)
            except Exception as exc:
                # Running out of daily tokens ends the run, and the cases that
                # already finished are kept. Any other failure is recorded
                # against that one case and the run carries on: a model that
                # will not produce a usable plan is a result, not a crash.
                if "rate_limit" in str(exc) or "429" in str(exc):
                    print(f'  STOP {case["id"]}: token budget exhausted')
                    break
                print(f'  ERR  {case["id"]}: {type(exc).__name__}')
                rows.append({"case_id": case["id"], "decision": "error", "amount": 0.0,
                             "cited": [], "retrieved": [], "faithfulness": None,
                             "seconds": 0.0})
                continue
            rows.append(row)
            ok = "ok  " if row["decision"] == case["expected_decision"] else "MISS"
            print(f'  {ok} {case["id"]}  expected {case["expected_decision"]}'
                  f'  got {row["decision"]}')

    run_pipeline("Naive RAG baseline", evaluate_case_naive, naive_rows)
    print()
    run_pipeline("ClauseCheck agent", evaluate_case_full, full_rows)

    if len(naive_rows) < len(eval_cases) or len(full_rows) < len(eval_cases):
        print(f"\nPARTIAL RUN: {len(naive_rows)} baseline and {len(full_rows)} agent "
              f"cases completed out of {len(eval_cases)}. Treat the table below as "
              "provisional.")

    payload = {"model": model_name, "cases": len(eval_cases),
               "run_at": datetime.now().isoformat(timespec="seconds"),
               "judge": JUDGE,
               "naive": {"summary": summarise(naive_rows, eval_cases), "rows": naive_rows},
               "full":  {"summary": summarise(full_rows, eval_cases),  "rows": full_rows}}
    # allow_nan=False so the file is valid JSON that any tool can read. A metric
    # with nothing to average is written as null rather than as a bare NaN.
    ablation_path.write_text(
        json.dumps(payload, indent=2, default=str, allow_nan=False), encoding="utf-8")
else:
    payload = json.loads(ablation_path.read_text(encoding="utf-8"))

print("\ndone")

### The ablation table

In [ ]:
LABELS = {"verdict_accuracy":  "Verdict accuracy",
          "amount_accuracy":   "Amount accuracy",
          "citation_accuracy": "Citation accuracy",
          "context_recall":    "Context recall",
          "faithfulness":      "Faithfulness",
          "abstention_rate":   "Abstention rate"}

rows = []
for key, label in LABELS.items():
    a = payload["naive"]["summary"].get(key)
    b = payload["full"]["summary"].get(key)
    fmt = lambda v: "n/a" if v is None or v != v else f"{v:.0%}"
    rows.append({"Metric": label, "Naive RAG": fmt(a), "ClauseCheck": fmt(b)})

print(f'model: {payload["model"]}   cases: {payload["cases"]}\n')
pd.DataFrame(rows).set_index("Metric")

In [ ]:
# Case by case, where each pipeline agrees with the label.
naive_by_id = {r["case_id"]: r for r in payload["naive"]["rows"]}
full_by_id  = {r["case_id"]: r for r in payload["full"]["rows"]}

comparison = []
evaluated_ids = set(naive_by_id) | set(full_by_id)
for case in [c for c in gold if c["id"] in evaluated_ids]:
    n, f = naive_by_id.get(case["id"], {}), full_by_id.get(case["id"], {})
    comparison.append({
        "case": case["id"],
        "tests": case["tests"][:36],
        "expected": case["expected_decision"],
        "naive": n.get("decision", "-"),
        "clausecheck": f.get("decision", "-"),
        "amount ok": "yes" if f and amount_is_correct(f, case) else "no",
    })

pd.DataFrame(comparison).set_index("case")

## 10. What we would say about this

**Where the gains come from.** The parts that constrain the model, not the parts that prompt it.
Clause-aware chunking keeps a rule intact so the grader reads a whole clause instead of half of
two. The metadata filter makes cross-policy contamination impossible rather than unlikely.
Moving the arithmetic into Python removes an error class that no prompt fixes. The baseline's
weakest metric is citation accuracy, and that is structural: fixed-size chunks carry no section
metadata, so it cannot reliably say where an answer came from even when the answer is right.

**Limits we would not hide.**

- The policy documents are generated samples. Real insurer PDFs have tables, two-column layouts
  and scanned pages, and the clause splitter will need work to survive them.
- Twenty scenarios is a small gold set. It shows a direction, not a confidence interval.
- The rules engine implements the common reading of proportionate deduction, applying the room
  rent ratio to the whole bill. Real policies differ on which heads of expenditure are caught.
- Faithfulness is judged by the same model that wrote the explanation, which is a known weakness
  of LLM-as-judge and a reason to read that number with suspicion.
- The re-ranker is not earning its place on a corpus this small. We kept it because it is part of
  the retrieval design and because it starts to matter as the corpus grows, but the measurement
  in section 4 says plainly that it is not what produced the numbers in section 9.

**Where it would go next.** Table extraction for the benefits schedule, a confidence signal on
each extracted parameter, and a second corpus to check that the clause splitter is not tuned to
one document's heading style.